# Image classification – prediction

In [ ]:
from pathlib import Path

import torch
from PIL import Image
from torch import nn
from torchvision import models, transforms
from torchvision.datasets.folder import IMG_EXTENSIONS

MODEL_PATH = "model.pt"
PREDICT_DIR = Path("data/predict")

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)  # ImageNet colour statistics


def load_image(path):
    image = Image.open(path).convert("RGBA")
    background = Image.new("RGBA", image.size, "white")  # transparent areas become white
    return Image.alpha_composite(background, image).convert("RGB")


def block(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
        nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
        nn.MaxPool2d(2),
    )


# same network as in 1_training.ipynb; the learned weights come from the model file
def build_model(pretrained, num_classes):
    if pretrained:
        model = models.resnet18()
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model
    return nn.Sequential(
        block(3, 32), block(32, 64), block(64, 128), block(128, 256),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, num_classes),
    )


checkpoint = torch.load(MODEL_PATH, map_location=device)
classes = checkpoint["classes"]
model = build_model(checkpoint["pretrained"], len(classes)).to(device)
model.load_state_dict(checkpoint["model_state"])
model.eval()

transform = transforms.Compose([
    transforms.Resize(checkpoint["image_size"]),
    transforms.CenterCrop(checkpoint["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
print("Classes:", ", ".join(classes))

In [ ]:
PREDICT_DIR.mkdir(parents=True, exist_ok=True)
paths = sorted(p for p in PREDICT_DIR.iterdir() if p.suffix.lower() in IMG_EXTENSIONS)
if not paths:
    print(f"Folder '{PREDICT_DIR}' is empty – add images and run this cell again.")

for path in paths:
    x = transform(load_image(path)).unsqueeze(0).to(device)
    with torch.no_grad():
        probabilities = model(x).softmax(1)[0]
    values, indices = probabilities.topk(min(3, len(classes)))
    print(path.name)
    for p, i in zip(values.tolist(), indices.tolist()):
        print(f"  {classes[i]:<30} {p:6.1%}")